In [236]:
import polars as pl
from pathlib import Path
import jellyfish
from nltk.corpus import names
import nltk
#nltk.download('names')
import re
import altair as alt

In [237]:
samples_df = pl.read_parquet("jails_pdfs_full.parquet")

In [238]:
yes_count = (samples_df['Resulting Death?'] == 'Yes').sum()


print (yes_count)

137


In [239]:
entries = ["Restraints Used\n", "Smattery"]


def clean_occurrences(entries):

    actual_occurrences = ["Suicide (method)", "Suicide (attempt)", "Homicide", "Homicide Attempt", "Escape", "Escape Attempt",
                "Fire", "Serious Injury", "Battery", "Riot of Rebellion", "Sex Offense", "Assault on Staff",
                "Assault among Detainees", "Fighting among Detainees", "Restraints Used", "OC Spray Used", "Other (specify)"]
    
    if entries.is_empty():
        return ["Error, no occurrence found"]

    cleaned_list = []
    for term in entries:
        best_choice = None
        highest_jaro = 0.0
        for compare_term in actual_occurrences:
            jaro_score = jellyfish.jaro_similarity(term, compare_term)
            if jaro_score > highest_jaro:
                highest_jaro = jaro_score
                best_choice = compare_term

        if best_choice in ["Suicide (method)", "Suicide (attempt)", "Other (specify)"]:
            # Keep the original term (possibly has colon text, like "Other (specify): Fight")
            cleaned_list.append(term)
        else:
            if best_choice not in cleaned_list:
                cleaned_list.append(best_choice)

    return cleaned_list

In [240]:
all_names = set(names.words())

test_list = ["Deaunte Campbell","3/12/2002","10/13/2021"]

def get_inmate_names(table_list):
    inmate_list = []
    pattern_punctuation = r"[^a-zA-Z0-9/ ]"
    for item in table_list:
        cleaned_item = re.sub(pattern_punctuation, '', item)
        tokens = cleaned_item.strip().split()
        for token in tokens:
            token = token.capitalize()
            if token in all_names:
                inmate_list.append(cleaned_item)
                break
    return inmate_list

get_inmate_names(test_list)

[]

In [241]:
occurrence_check = samples_df.select(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))

C:\Users\ganon\AppData\Local\Temp\ipykernel_20624\2675670731.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  occurrence_check = samples_df.select(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))


In [242]:
samples_df = samples_df.with_columns(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))

C:\Users\ganon\AppData\Local\Temp\ipykernel_20624\3155367749.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  samples_df = samples_df.with_columns(pl.col("Occurrence").map_elements(clean_occurrences).alias("Cleaned Occurrences"))


In [243]:
samples_df = samples_df.with_columns(pl.col("Table Contents").map_elements(get_inmate_names).alias("Inmate Names"))

C:\Users\ganon\AppData\Local\Temp\ipykernel_20624\3131983131.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  samples_df = samples_df.with_columns(pl.col("Table Contents").map_elements(get_inmate_names).alias("Inmate Names"))


In [244]:
# Clean facility name:
def clean_facility_name(df):
    """Clean facility names using Polars expressions"""
    return df.with_columns(
        pl.col('Facility Name')
        # First, clean OCR artifacts
        .str.replace_all(r'\\n|\\\\|\\/', ' ') 
        .str.replace_all(r'\n', ' ')
        .str.replace_all(r'[\\\/]+', ' ')
        .str.replace_all(r'\s+', ' ') 
        .str.strip_chars()
        # Extract facility name after any variation of "Name:"
        .str.extract(r'cility Name:\s*(.+)', 1)
        # Additional cleanup of extracted names
        .str.replace_all(r'[^a-zA-Z\s\'-]', ' ')  # Keep only letters, spaces, apostrophes, hyphens
        .str.replace_all(r'\s+', ' ')  # Clean multiple spaces again
        .str.strip_chars()  # Final trim
        # Stop at any word ending in "ddres" and remove that word + everything after
        .str.replace_all(r'\s+(ddress|ddrace).*$', '')
        .str.replace_all(r"(?i)\bSheriff's\b", "Sheriffs")
        .str.to_titlecase()
        .alias('Cleaned Facility Name')
    )

samples_df = clean_facility_name(samples_df)

# Date cleanning
def clean_date_occurrence(df):
    return df.with_columns(
        pl.col("Date")
          # OCR cleanup (keep forward slashes)
          .str.replace_all(r"\\n|\\\\", " ")
          .str.replace_all(r"\n", " ")
          .str.replace_all(r"\s+", " ")
          .str.strip_chars()
          # Extract the MM/DD/YYYY text
          .str.extract(r"Date of Occurrence:\s*([0-9]{1,2}/[0-9]{1,2}/(20[0-9]{2}))", 1)
          # Parse & re‐format to zero‐padded MM/DD/YYYY
          .str.strptime(pl.Date, "%m/%d/%Y", strict=False)
          .alias("temp_date")
    ).with_columns(
        # Only keep dates between 2000 and 2025, otherwise set to null
        pl.when(pl.col("temp_date").dt.year().is_between(2000, 2025))
        .then(pl.col("temp_date").dt.strftime("%m/%d/%Y"))
        .otherwise(None)
        .alias("Cleaned Date")
    ).drop("temp_date")

samples_df = clean_date_occurrence(samples_df)

In [ ]:
samples_df = samples_df.drop(["OCR_Confidence", "OCR_Word_Count", "OCR_Text_Detected", "Occurrence", "Facility Name", "Date", "Cook County?"])

In [248]:
samples_df = samples_df.with_columns(pl.col("Cleaned Occurrences").list.join(",").alias("Cleaned Occurrences"))

samples_df = samples_df.with_columns(pl.col("Inmate Names").list.join(",").alias("Inmate Names"))

samples_df = samples_df.with_columns(pl.col("Table Contents").list.join(",").alias("Table Contents"))

In [249]:
samples_df = samples_df.select(["Report ID", "Facility Type", "Cleaned Facility Name", "Address", "Phone Number", "Cleaned Date", "Time of Day", "AM or PM", "Table Contents", "Inmate Names", "Cleaned Occurrences", "Injuries?", "Resulting Death?", 'Deceased Cause, Date, and Time', 'Deceased on Suicide Watch', 'Deceased Reporter', 'Deceased Examined by Physician', 'Deceased Signs of Illness'])

In [250]:
samples_df = samples_df.with_columns(
    pl.when(pl.col("Cleaned Occurrences") == "")
    .then(pl.lit("Error, no occurrence found"))
    .otherwise(pl.col("Cleaned Occurrences"))
    .alias("Cleaned Occurrences")
)

def clean_other(df):
    pattern = r"specify"
    df = df.with_columns(
    pl.when(pl.col("Cleaned Occurrences").str.contains(pattern))
    .then(pl.col("Cleaned Occurrences").str.extract(r"specify(.*)", group_index=1))
    .otherwise(pl.col("Cleaned Occurrences"))
    .alias("Cleaned Occurrences")
    )

    return df

total_occurrence_result = (
    samples_df
    .with_columns(
        pl.col("Cleaned Occurrences").str.split(",").alias("Cleaned Occurrences")
    )
    .explode("Cleaned Occurrences")
    .filter(pl.col("Cleaned Occurrences").is_not_null())
    .select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(12)
)

total_occurrence_result = clean_other(total_occurrence_result)

total_occurrence_result = total_occurrence_result.filter(~pl.col("Cleaned Occurrences").is_in(["Error", " no occurrence found"]))

total_occurrence_result

Cleaned Occurrences,count
str,u32
"""Fighting among Detainees""",6383
"""Battery""",5590
"""Restraints Used""",3630
"""OC Spray Used""",2894
"""Assault on Staff""",975
"""Sex Offense""",829
"""Serious Injury""",596
"""Homicide Attempt""",407
"""Other: OTHER — WEAPON """,389


In [279]:
top_prisons = (
    samples_df.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(11)
    )

top_prisons = top_prisons.filter(pl.col("Cleaned Facility Name") != "")

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Missing 2,239 County Names (8.19%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(-50),
    y=alt.value(230),  # just below the chart (chart height + some offset)
    text="text:N"
)

top_prisons_chart = alt.Chart(top_prisons).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Occurrence Counts by Jails"
)

# Use layering to draw the footnote over/under the chart
top_prisons_chart = alt.layer(
    top_prisons_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

top_prisons_chart

alt.LayerChart(...)

In [273]:
top_prisons

Cleaned Facility Name,count
str,u32
"""Cook County Department Of Corr…",11071
null,2239
"""Kane County Sheriffs Office""",1486
"""Madison County Sheriffs Office""",1262
"""Lake County Jail""",1167
…,…
"""Williamson County Jail""",215
"""Tazewell County Jail""",207
"""Coles County Safety Detention …",152


In [261]:
total_occurrence_chart = alt.Chart(total_occurrence_result).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
).properties(
    title="Most Common Occurrences Across All Jails"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Occurrences may be concurrent. Missing 3,418 Occurrences (12.5%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(-50),
    y=alt.value(230),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
total_occurrence_chart = alt.layer(
    total_occurrence_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

total_occurrence_chart

alt.LayerChart(...)

In [266]:
pattern = r"\bspecify\b"

other_occurrence_table = samples_df.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)
    
other_occurrence_table = clean_other(other_occurrence_table)

other_occurrence_table = (
    other_occurrence_table.select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(10)
)

other_occurrence_chart = alt.Chart(other_occurrence_table).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
).properties(
    title="Most Common Other Occurrences Across All Jails"
)

In [267]:
other_occurrence_chart

alt.Chart(...)

In [254]:
pattern = r"\bRestraints\b"

restraints_table = samples_df.filter(
    pl.col("Cleaned Occurrences").str.contains(pattern)
)

restraints_table = (
    restraints_table.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(21)
)

restraints_table = restraints_table.filter(pl.col("Cleaned Facility Name") != "null")

restraints_chart = alt.Chart(restraints_table).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Facility Name:N", title="Jail").sort('-x')
).properties(
    title="Top Jails for Restraint Usage"
)

# Footnote as a separate text chart
footnote = alt.Chart(pl.DataFrame({
    "text": ["Note: Missing 214 Jail Names (12%)"],
    "x": [0],
    "y": [0]
})).mark_text(
    align="left",
    baseline="bottom",
    fontSize=10,
    dx=0,
    dy=20,  # distance below the main chart area
    color="gray"
).encode(
    x=alt.value(0),
    y=alt.value(430),  # just below the chart (chart height + some offset)
    text="text:N"
)

# Use layering to draw the footnote over/under the chart
restraints_chart = alt.layer(
    restraints_chart,
    footnote
).configure_view(
    stroke=None  # removes box around the chart
)

In [255]:
restraints_chart

alt.LayerChart(...)

In [256]:
samples_df

Report ID,Facility Type,Cleaned Facility Name,Address,Phone Number,Cleaned Date,Time of Day,AM or PM,Table Contents,Inmate Names,Cleaned Occurrences,Injuries?,Resulting Death?,"Deceased Cause, Date, and Time",Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Sampson, Kevin 1-12 (suicide a…","""ant. Inc ""","""Latthsons Otice Bn Fapameat Tt…","""Street City State Zip Code ""","""elephone #: _70$ -St3~ 3) 3c _…",null,"""ime of Occurrence: _O4 Se phan…","""Error: less than two boxes fou…","""WELIIIC CS WIVOlVEU,wae eee,Ke…","""Ae bel pIvVAriie""","""Error, no occurrence found""","""Yes, attach coroner's report o…","""Error: less than two boxes fou…","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""Cross, Anton 1-27 (suicide att…","""County ""","""Kane County Sheriffs Office""","""\ddress: 37W755 IL Route 38 Su…","""__ Telephone #: 630-232-6840 ""","""01/27/2021""","""_ Time of Occurrence: 204: ""","""No filled box found""","""Cross, Anton Nelson,| 02/04/19…","""Cross Anton Nelson""","""Suicide Attempt (method) Drink…","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""Webster, Tiffany 1-9_p1""","""County ""","""Mchenry County Adult Correctio…","""Address: 2200 N. Seminary Ave.…","""__ Telephone #: 815-338-4741 _…","""01/09/2021""","""“ wy wiew Time of Occurrence: …","""p.m. ""","""vee ence,Webster, Tiffany,0 I …","""Webster Tiffany""","""Error, no occurrence found""","""No injuries""","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""Magana, Yvette 1-14_p1""","""County ""","""Williamson County Jail""","""\ddress: 404 N. Van Buren Mari…","""___ Telephone #: 618-998-2100 ""","""01/14/2021""","""_ Time of Occurrence: 235 ""","""No filled box found""",""". 01/14/2021,Yvette Magana,09/…","""Yvette Magana""","""Restraints Used,Assault on Sta…","""Yes, (briefly describe): CO J.…","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""Huff, Anthony 1-15_p1""","""County ""","""Lake County Adult Corrections""","""Address: 29 South County Stree…","""_. Telephone #: 847 377-4099 ""","""01/15/2021""","""_ Time of Occurrence: 2:50 ""","""pm, ""","""-10n8/ 1984,09/25/20 | a,_ Thr…","""Hut Anthony L160833""","""ther (specify): Sick Person ""","""Yes, (briefly describe): Inmat…","""No ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""UO - JDSU (April 2022)_p537""","""Error: less than two boxes fou…","""Cook County Department Of Corr…","""Aqaress: SIUM YS SGM MAIVIOIVE…","""Telephone #: 773.674.3474 O IL…","""04/14/2022""","""SSS WE Ow EE Eee Ne NN ON l1 G…","""Check time, no AM or PM for Co…","""Adebowale Awolola,4/14/1995,9/…","""John Hurt JR""","""Battery""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""UO - JDSU (April 2022)_p538""","""Error: less than two boxes fou…","""Cook County Department Of Corr…","""Aqaress: SIUM Y- SGM MIiVioiVi…","""Telephone #: 773.674.3474 O IL…","""04/06/2022""","""SSS WE Ow EE Eee omen Te NE Tl…","""Check time, no AM or PM for Co…","""Steven Bailey,10/14/1995,10/2/…","""Steven Bailey,Keith Perkins,Dr…","""Fighting among Detainees""","""No injuries""","""NC ""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""
"""UO - JDSU (April 2022)_p539""","""Error: less than two boxes fou…","""Cook County Department Of Corr…","""Aqaress: SIUM Y4- SGM MIVioIVt…","""Telephone #: 773.674.3474 O IL…","""04/05/2022""","""SSS WE Ow EE Eee a Nr er Tl) G…","""Check time, no AM or PM for Co…","""ARMONI ARRINGTON,12/24/2002,1/…","""Dorian R Childs""","""Fighting among Detainees""","""No injuries""","""No filled box found""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A"""


In [257]:
top_counties = (samples_df.select(
        pl.col("Cleaned Facility Name").value_counts(sort=True)
    )
    .unnest("Cleaned Facility Name")
    .head(11)
)

top_counties = top_counties.filter(pl.col("Cleaned Facility Name") != "")

#2239 missing county names, 8.19% missing

def get_county_occurrences(current_jail):
    filtered_df = samples_df.filter(pl.col("Cleaned Facility Name") == current_jail)
    current_jail_result = (
    filtered_df.select(
        pl.col("Cleaned Occurrences").value_counts(sort=True)
    )
    .unnest("Cleaned Occurrences")
    .head(11)
    )
    current_jail_result = current_jail_result.filter(pl.col("Cleaned Occurrences") != "Error, no occurrence found")
    current_jail_chart = alt.Chart(current_jail_result).mark_bar().encode(
    x=alt.X("count:Q", title="Count"),
    y=alt.Y("Cleaned Occurrences:N", title="Occurrence").sort('-x')
    ).properties(
        title=current_jail
    )
    return current_jail_chart

cook_county = get_county_occurrences("Cook County Department Of Corrections")

kane_county = get_county_occurrences("Kane County Sheriffs Office")

madison_county = get_county_occurrences("Madison County Sheriffs Office")


In [258]:
cook_county

alt.Chart(...)

In [259]:
top_counties

Cleaned Facility Name,count
str,u32
"""Cook County Department Of Corr…",11071
"""Kane County Sheriffs Office""",1486
"""Madison County Sheriffs Office""",1262
"""Lake County Jail""",1167
"""Peoria County Sheriffs Office""",918
"""Sangamon County Jail""",733
"""Dupage County Jail""",408
"""St Clair County Sheriffs Depar…",391
"""Macon County Jail""",350


In [260]:
#samples_df.write_excel("7_10_sample.xlsx")

To-Do:
- Case Study of One Individual
- Breakdown of largest counties